# `apply_linear_transform()`

The grid utility `nematics3d.apply_linear_transform()` maps three-dimensional lattice-index coordinates to physical coordinates, or applies the inverse map. `Nematics3D` uses the row-vector convention

$$\mathbf{x}_{\mathrm{physical}} = \mathbf{x}_{\mathrm{index}} T + \mathbf{o},$$

where $T$ is `transform` and $\mathbf{o}$ is `offset`. Three facts are important from the beginning:

- the final axis of `points` must have length 3, while any leading shape is preserved;
- a non-identity transform must be a finite, right-handed $3\times3$ matrix with three nonzero, pairwise-orthogonal rows; independent row scales are allowed, but shear and reflection are not;
- `is_inv=True` reverses both the translation and the linear transform, so a forward/inverse round trip recovers the original coordinates up to floating-point precision.

## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** The following cell imports `NumPy` and `Nematics3D`.

In [1]:
import numpy as np
import nematics3d as n3d

## Minimal example: scale and translate one point

A diagonal transform scales the lattice axes independently. The offset is added after the scaling.

In [2]:
point_index = np.array([1.0, 2.0, 3.0])
transform = np.diag([2.0, 3.0, 4.0])
offset = np.array([10.0, 20.0, 30.0])

point_physical = n3d.apply_linear_transform(
    point_index,
    transform=transform,
    offset=offset,
)
print(point_physical)

[12. 26. 42.]


## Inputs and outputs

The public signature is:

```python
apply_linear_transform(
    points,
    transform=GRID_TRANSFORM_IDENTITY,
    offset=None,
    *,
    is_inv=False,
)
```

### `points`

`points` contains real, finite three-dimensional coordinates. The final axis is the coordinate axis; all preceding axes are retained.

| Example shape | Meaning | Output shape |
| --- | --- | --- |
| `(3,)` | One point | `(3,)` |
| `(N, 3)` | A point collection or trajectory | `(N, 3)` |
| `(Nx, Ny, Nz, 3)` | A complete coordinate grid | `(Nx, Ny, Nz, 3)` |
| `(0, 3)` | An empty point collection | `(0, 3)` |

Lists, tuples, and `NumPy` arrays are accepted. Boolean, string, complex, `NaN`, and infinite coordinates are rejected, as are inputs whose trailing axis does not have length 3. The returned floating-point array is independent of the input.

### `transform`

`transform` accepts `None`, `GRID_TRANSFORM_IDENTITY`, or a real finite `(3, 3)` array. `None` and the sentinel are normalized to the same canonical identity object. A matrix input must satisfy all of the following conditions:

- every row is nonzero;
- distinct rows are orthogonal within numerical tolerance;
- the determinant is positive, so the basis is right-handed;
- row lengths may differ, allowing anisotropic grid spacing.

`GridTransform` is the reader-facing semantic alias used in signatures. `as_grid_transform()` enforces the runtime contract.

### `offset` and `is_inv`

`offset` is either `None` or one real, finite three-dimensional vector. It has no separate semantic alias; `as_grid_offset()` performs runtime validation. `is_inv=False` applies the forward map, while `is_inv=True` applies the inverse. Boolean options accept booleans and numeric zero or one; other values are rejected.

## Examples

### Rotation with anisotropic scale

The following matrix rotates in the xy plane and assigns separate lengths to its three orthogonal rows. Each row is the physical basis vector reached by one unit step along the corresponding lattice-index axis.

In [3]:
theta = np.pi / 4
rotation = np.array(
    [
        [np.cos(theta), -np.sin(theta), 0.0],
        [np.sin(theta), np.cos(theta), 0.0],
        [0.0, 0.0, 1.0],
    ]
)
scaled_rotation = np.diag([2.0, 3.0, 4.0]) @ rotation.T
validated = n3d.as_grid_transform(scaled_rotation)

points = np.eye(3)
print(n3d.apply_linear_transform(points, validated))

[[ 1.41421356  1.41421356  0.        ]
 [-2.12132034  2.12132034  0.        ]
 [ 0.          0.          4.        ]]


### Preserve a multidimensional grid shape

Only the final coordinate axis participates in the matrix multiplication. The leading `(2, 2)` grid shape is unchanged.

In [4]:
grid_index = np.arange(12.0).reshape(2, 2, 3)
grid_physical = n3d.apply_linear_transform(
    grid_index,
    transform=np.diag([2.0, 3.0, 4.0]),
)
print("input shape:", grid_index.shape)
print("output shape:", grid_physical.shape)
print(grid_physical)

input shape: (2, 2, 3)
output shape: (2, 2, 3)
[[[ 0.  3.  8.]
  [ 6. 12. 20.]]

 [[12. 21. 32.]
  [18. 30. 44.]]]


### Apply the inverse map

The inverse first removes the offset and then solves the linear system.

In [5]:
restored = n3d.apply_linear_transform(
    point_physical,
    transform=transform,
    offset=offset,
    is_inv=True,
)
print(restored)
print("round trip agrees:", np.allclose(restored, point_index))

[1. 2. 3.]
round trip agrees: True


### Store validated parameters as read-only snapshots

High-level objects use `is_readonly=True` when transform parameters become internal state. The returned arrays are independent snapshots: changing the original arrays does not alter the stored values.

In [6]:
source_transform = np.diag([2.0, 3.0, 4.0])
stored_transform = n3d.as_grid_transform(
    source_transform,
    is_readonly=True,
)
source_transform[0, 0] = 99.0

print(stored_transform)
print("writeable:", stored_transform.flags.writeable)

[[2. 0. 0.]
 [0. 3. 0.]
 [0. 0. 4.]]
writeable: False


## Details

### Forward and inverse order

For the forward map, each row-vector coordinate is multiplied on the right by $T$, then $\mathbf{o}$ is added. Row $i$ of $T$ is therefore the physical basis vector associated with one unit step along lattice-index axis $i$. Matrix order matters: replacing $T$ with $T^{\mathsf{T}}$ generally describes a different map.

The inverse relation is

$$\mathbf{x}_{\mathrm{index}} T = \mathbf{x}_{\mathrm{physical}} - \mathbf{o}.$$

The implementation solves this system with `numpy.linalg.solve()` rather than explicitly constructing $T^{-1}$.

### Validation and the identity fast path

`as_grid_transform()` checks shape, finite values, nonzero row lengths, orthogonality, and handedness. `None` and `GRID_TRANSFORM_IDENTITY` are canonicalized to the unique sentinel, allowing transformation code to bypass matrix multiplication when no linear transform is present. Offset-only maps still add or subtract $\mathbf{o}$.

`is_readonly=True` is intended for object state. It creates an independent array and disables in-place writes; it does not make the caller's original array read-only.

## Possible issues

### A general affine matrix is rejected

The grid convention deliberately excludes shear, reflection, and degenerate axes. `apply_linear_transform()` is not a general-purpose affine-transformation API.

### The matrix appears transposed

Check the row-vector formula at the top of this tutorial. Code or references using column vectors often write $T\mathbf{x}$ instead of $\mathbf{x}T$ and therefore store the corresponding matrix transposed.

### A read-only result cannot be edited

Use the default `is_readonly=False` for a normal writable validated array. Read-only mode is for stable internal snapshots; create a copy before intentionally editing one.

### Very small grid scales

Transforms with a row length at or below the degeneracy tolerance are rejected. Orthogonality and handedness are checked after normalizing the row directions, so otherwise valid transforms are treated consistently across physical scales.

## Where the transform utilities are used

`Nematics3D` uses these utilities when constructing shared grids, converting defect-line indices into physical coordinates, wrapping transformed points into periodic boxes, generating contour surfaces, and mapping between index-space and physical-space queries. `GridFieldDataset`, `QFieldObject`, and `DisclinationLine` store validated transform state as read-only snapshots.

## Useful Links

### Referenced in this tutorial

- [Grid-transform source](../../../src/nematics3d/grid/transform.py) — defines `GridTransform`, validation, immutable storage, identity detection, and forward/inverse application.
- [`as_tensor()` source](../../../src/nematics3d/datatypes/tensor.py) — validates the transform's real finite `(3, 3)` array structure.
- [`as_vector()` source](../../../src/nematics3d/datatypes/vector.py) — validates the three-dimensional offset.
- [`as_points()` source](../../../src/nematics3d/datatypes/points.py) — validates transformed coordinates while preserving their leading shape.
- [Periodic grid utilities](../periodic/unwrap_trajectory.ipynb) — demonstrates a downstream workflow that also uses grid-coordinate conventions.

### Going deeper

- [`defect_classify_into_lines()` tutorial](../../analysis/disclination/defect_classify_into_lines.ipynb) — constructs line objects carrying a grid transform and offset.
- [`QFieldObject` initialization](../../classes/QFieldObject/initialize.ipynb) — configures a high-level $Q$-field object and its spatial grid.